# ID Minter Invoke And Verify Harness

This notebook samples root work source identifiers from `works-source-2025-10-02`, invokes the new id_minter Lambda, reads back results from `works-identified-2026-03-06`, and compares them with the current `works-identified-2025-10-02` baseline index.

Verification is done via the Scala decode checker:

- **Matcher decode**: decodes target docs as `WorkStub` (matcher-level compatibility)
- **Merger decode**: decodes target docs as `Work[Identified]` (full compatibility)
- **Compare**: decodes both target and baseline docs as `Work[Identified]` and checks they produce identical Scala objects — this proves differences like extra type fields or explicit nulls in the Python output are transparent to downstream Scala consumers

Run the cells in order and start with the smoke test before moving to the 1k verification batch.

In [72]:
import json
import os
import random
import subprocess
import sys
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Any, NoReturn

import boto3
from botocore.exceptions import UnauthorizedSSOTokenError

SRC_ROOT = Path.cwd().resolve().parent / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    import pandas as pd
except ImportError:
    pd = None

from utils.aws import dicts_from_s3_jsonl
from utils.elasticsearch import get_client, get_standard_index_name
from utils.models.manifests import StepManifest

os.environ.setdefault("AWS_PROFILE", "platform-developer")

AWS_PROFILE = os.environ["AWS_PROFILE"]
REGION = "eu-west-1"
PIPELINE_DATE = "2025-10-02"
TARGET_INDEX_DATE_SUFFIX = "2026-03-06"
LAMBDA_ARN = "arn:aws:lambda:eu-west-1:760097843905:function:catalogue-2025-10-02-id-minter"
SOURCE_INDEX = get_standard_index_name("works-source", PIPELINE_DATE)
BASELINE_IDENTIFIED_INDEX = get_standard_index_name("works-identified", PIPELINE_DATE)
TARGET_IDENTIFIED_INDEX = get_standard_index_name("works-identified", TARGET_INDEX_DATE_SUFFIX)
VERIFICATION_BATCH_SIZE = 1000
DEFAULT_SAMPLE_BUFFER = 250
REPO_ROOT = Path.cwd().resolve().parents[1]
SCALA_DECODE_MAIN_CLASS = "weco.pipeline.merger.tools.IdentifiedIndexDecodeChecker"
SCALA_DECODE_DIR = REPO_ROOT / ".scratch" / "id_minter_decode_checks"
SCALA_DECODE_DIR.mkdir(parents=True, exist_ok=True)

session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
lambda_client = session.client("lambda", region_name=REGION)
_source_es_client = None
_baseline_es_client = None
_target_es_client = None


def raise_sso_login_error(error: UnauthorizedSSOTokenError) -> NoReturn:
    raise RuntimeError(
        f"AWS SSO session for profile {AWS_PROFILE} has expired. "
        f"Run `aws sso login --profile {AWS_PROFILE}` and re-run the cell."
    ) from error


def get_source_es_client() -> Any:
    global _source_es_client
    if _source_es_client is None:
        try:
            _source_es_client = get_client("id_minter", PIPELINE_DATE, "public")
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)
    return _source_es_client


def get_baseline_es_client() -> Any:
    global _baseline_es_client
    if _baseline_es_client is None:
        try:
            _baseline_es_client = get_client("id_minter", PIPELINE_DATE, "public")
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)
    return _baseline_es_client


def get_target_es_client() -> Any:
    global _target_es_client
    if _target_es_client is None:
        try:
            _target_es_client = get_client("id_minter", PIPELINE_DATE, "public")
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)
    return _target_es_client


RUN_PREFIX = f"id-minter-verify-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

print(f"AWS profile:             {AWS_PROFILE}")
print(f"Lambda ARN:              {LAMBDA_ARN}")
print(f"Source index:            {SOURCE_INDEX}")
print(f"Baseline identified:     {BASELINE_IDENTIFIED_INDEX}")
print(f"Target identified:       {TARGET_IDENTIFIED_INDEX}")
print(f"Verification batch size: {VERIFICATION_BATCH_SIZE}")
print(f"Run prefix:              {RUN_PREFIX}")
print(f"Repo root:               {REPO_ROOT}")
print("Elasticsearch clients are created lazily when sampling or comparing.")

AWS profile:             platform-developer
Lambda ARN:              arn:aws:lambda:eu-west-1:760097843905:function:catalogue-2025-10-02-id-minter
Source index:            works-source-2025-10-02
Baseline identified:     works-identified-2025-10-02
Target identified:       works-identified-2026-03-06
Verification batch size: 1000
Run prefix:              id-minter-verify-20260313T103459Z
Repo root:               /Users/kennyr/workspace/catalogue-pipeline
Elasticsearch clients are created lazily when sampling or comparing.


In [73]:
def build_source_identifier_string(source_identifier: dict[str, Any]) -> str:
    return (
        f"{source_identifier['ontologyType']}"
        f"[{source_identifier['identifierType']['id']}/{source_identifier['value']}]"
    )


def build_job_id(label: str | None = None) -> str:
    suffix = uuid.uuid4().hex[:8]
    if label:
        return f"{RUN_PREFIX}-{label}-{suffix}"
    return f"{RUN_PREFIX}-{suffix}"


def manifest_location_to_s3_uri(location: Any) -> str:
    return f"s3://{location.bucket}/{location.key}"


def read_manifest_batches(manifest: StepManifest | None) -> dict[str, list[dict[str, Any]]]:
    if manifest is None:
        return {"successes": [], "failures": []}

    try:
        batches = {
            "successes": dicts_from_s3_jsonl(
                manifest_location_to_s3_uri(manifest.successes.batch_file_location)
            )
        }
        if manifest.failures is None:
            batches["failures"] = []
        else:
            batches["failures"] = dicts_from_s3_jsonl(
                manifest_location_to_s3_uri(manifest.failures.error_file_location)
            )
        return batches
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)


def extract_root_source_identifier(doc_source: dict[str, Any]) -> dict[str, Any] | None:
    return doc_source.get("state", {}).get("sourceIdentifier")


def flatten_manifest_canonical_ids(
    manifest_batches: dict[str, list[dict[str, Any]]],
) -> list[str]:
    canonical_ids: list[str] = []
    for line in manifest_batches["successes"]:
        canonical_ids.extend(line.get("canonicalIds", []))
    return canonical_ids


def fetch_docs_by_canonical_id(
    es_client: Any,
    index: str,
    canonical_ids: list[str],
    source_fields: list[str] | None = None,
) -> dict[str, Any]:
    unique_ids = list(dict.fromkeys(canonical_ids))
    started = perf_counter()
    docs_by_canonical_id: dict[str, dict[str, Any]] = {}

    if not unique_ids:
        return {
            "docs_by_canonical_id": docs_by_canonical_id,
            "query_count": 0,
            "elapsed_seconds": perf_counter() - started,
            "missing_canonical_ids": [],
        }

    try:
        kwargs: dict[str, Any] = {
            "index": index,
            "body": {"ids": unique_ids},
        }
        if source_fields is not None:
            kwargs["_source"] = source_fields
        response = es_client.mget(**kwargs)
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)

    docs_by_canonical_id = {
        doc["_id"]: doc
        for doc in response["docs"]
        if doc.get("found") and doc.get("_source")
    }
    missing_ids = [cid for cid in unique_ids if cid not in docs_by_canonical_id]

    return {
        "docs_by_canonical_id": docs_by_canonical_id,
        "query_count": 1,
        "elapsed_seconds": perf_counter() - started,
        "missing_canonical_ids": missing_ids,
    }


def sample_source_work_candidates(
    desired_count: int,
    oversample_buffer: int = DEFAULT_SAMPLE_BUFFER,
    exclude_today: bool = True,
    seed: int | None = None,
) -> list[dict[str, Any]]:
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    seed = random.randint(0, 2**31) if seed is None else seed
    try:
        response = get_source_es_client().search(
            index=SOURCE_INDEX,
            body={
                "_source": ["state.sourceIdentifier", "state.sourceModifiedTime"],
                "query": {
                    "function_score": {
                        "query": {"match_all": {}},
                        "random_score": {"seed": seed, "field": "_seq_no"},
                    }
                },
                "size": desired_count + oversample_buffer,
            },
        )
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)

    candidates: list[dict[str, Any]] = []
    skipped = Counter()
    seen_source_identifiers: set[str] = set()

    for hit in response["hits"]["hits"]:
        source_doc = hit["_source"]
        state = source_doc.get("state", {})
        source_identifier = state.get("sourceIdentifier")
        if not source_identifier:
            skipped["missing_root_source_identifier"] += 1
            continue

        source_identifier_str = build_source_identifier_string(source_identifier)
        if source_identifier_str in seen_source_identifiers:
            skipped["duplicate_root_source_identifier"] += 1
            continue
        source_modified_time = state.get("sourceModifiedTime", "")
        if exclude_today and source_modified_time.startswith(today):
            continue

        seen_source_identifiers.add(source_identifier_str)
        candidates.append(
            {
                "source_doc_id": hit["_id"],
                "source_identifier": source_identifier,
                "source_identifier_str": source_identifier_str,
                "source_modified_time": source_modified_time,
            }
        )
        if len(candidates) >= desired_count:
            break

    print(
        f"Sampled {len(candidates)} candidates from {SOURCE_INDEX} "
        f"(seed={seed}, skipped={dict(skipped)})"
    )
    return candidates


def invoke_id_minter(
    source_identifiers: list[str],
    label: str | None = None,
) -> dict[str, Any]:
    job_id = build_job_id(label)
    payload = {
        "source_identifiers": source_identifiers,
        "job_id": job_id,
    }

    started = perf_counter()
    try:
        response = lambda_client.invoke(
            FunctionName=LAMBDA_ARN,
            InvocationType="RequestResponse",
            Payload=json.dumps(payload).encode("utf-8"),
        )
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)
    elapsed_seconds = perf_counter() - started
    raw_payload = response["Payload"].read().decode("utf-8")

    parsed_payload: dict[str, Any] | None
    if raw_payload.strip():
        parsed_payload = json.loads(raw_payload)
    else:
        parsed_payload = None

    manifest = None
    if response.get("FunctionError") is None and parsed_payload is not None:
        manifest = StepManifest.model_validate(parsed_payload)

    return {
        "job_id": job_id,
        "request_payload": payload,
        "status_code": response["StatusCode"],
        "function_error": response.get("FunctionError"),
        "elapsed_seconds": elapsed_seconds,
        "raw_payload": raw_payload,
        "parsed_payload": parsed_payload,
        "manifest": manifest,
    }


def run_single_batch(
    candidates: list[dict[str, Any]],
    label: str | None = None,
) -> dict[str, Any]:
    timings_started = perf_counter()
    source_identifiers = [candidate["source_identifier_str"] for candidate in candidates]

    invocation = invoke_id_minter(source_identifiers, label=label)

    manifest_started = perf_counter()
    manifest_batches = read_manifest_batches(invocation["manifest"])
    manifest_elapsed_seconds = perf_counter() - manifest_started

    target_canonical_ids = flatten_manifest_canonical_ids(manifest_batches)
    target_fetch = fetch_docs_by_canonical_id(
        get_target_es_client(), TARGET_IDENTIFIED_INDEX, target_canonical_ids,
    )
    baseline_fetch = fetch_docs_by_canonical_id(
        get_baseline_es_client(), BASELINE_IDENTIFIED_INDEX, target_canonical_ids,
    )

    batch_summary = {
        "job_id": invocation["job_id"],
        "batch_size": len(candidates),
        "status_code": invocation["status_code"],
        "function_error": invocation["function_error"],
        "lambda_seconds": round(invocation["elapsed_seconds"], 3),
        "manifest_seconds": round(manifest_elapsed_seconds, 3),
        "target_fetch_seconds": round(target_fetch["elapsed_seconds"], 3),
        "baseline_fetch_seconds": round(baseline_fetch["elapsed_seconds"], 3),
        "total_post_lambda_seconds": round(
            perf_counter() - timings_started - invocation["elapsed_seconds"],
            3,
        ),
        "manifest_success_count": invocation["manifest"].successes.count
        if invocation["manifest"] is not None
        else 0,
        "manifest_failure_count": invocation["manifest"].failures.count
        if invocation["manifest"] is not None and invocation["manifest"].failures is not None
        else 0,
        "success_batch_line_count": len(manifest_batches["successes"]),
        "failure_batch_line_count": len(manifest_batches["failures"]),
        "target_manifest_canonical_ids": len(target_canonical_ids),
        "target_docs_found": len(target_fetch["docs_by_canonical_id"]),
        "target_missing_docs": len(target_fetch["missing_canonical_ids"]),
        "baseline_docs_found": len(baseline_fetch["docs_by_canonical_id"]),
        "baseline_missing_docs": len(baseline_fetch["missing_canonical_ids"]),
    }

    return {
        "summary": batch_summary,
        "invocation": invocation,
        "manifest_batches": manifest_batches,
        "target_fetch": target_fetch,
        "baseline_fetch": baseline_fetch,
    }


def run_verification_batches(
    candidate_pool: list[dict[str, Any]],
    batch_size: int = VERIFICATION_BATCH_SIZE,
    max_batches: int | None = None,
) -> dict[str, Any]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    available_batches = len(candidate_pool) // batch_size
    if available_batches == 0:
        raise ValueError(
            f"Need at least {batch_size} candidates, only have {len(candidate_pool)}"
        )

    batch_count = available_batches if max_batches is None else min(max_batches, available_batches)
    runs = []
    for batch_index in range(batch_count):
        start = batch_index * batch_size
        batch_candidates = candidate_pool[start : start + batch_size]
        runs.append(
            run_single_batch(
                batch_candidates,
                label=f"verify-{batch_size}-{batch_index + 1:02d}",
            )
        )

    metrics = [run["summary"] for run in runs]
    return {"runs": runs, "metrics": metrics}

In [74]:
MATCHER_SOURCE_FIELDS = ["state", "version", "type"]


def get_batch_runs(result: dict[str, Any]) -> list[dict[str, Any]]:
    if "runs" in result:
        return result["runs"]
    return [result]


def collect_manifest_canonical_ids(result: dict[str, Any]) -> list[str]:
    canonical_ids: list[str] = []
    for batch_run in get_batch_runs(result):
        canonical_ids.extend(flatten_manifest_canonical_ids(batch_run["manifest_batches"]))
    return list(dict.fromkeys(canonical_ids))


def collect_target_docs_by_canonical_id(
    result: dict[str, Any],
) -> dict[str, dict[str, Any]]:
    canonical_ids = collect_manifest_canonical_ids(result)
    docs_by_canonical_id: dict[str, dict[str, Any]] = {}
    for batch_run in get_batch_runs(result):
        batch_docs = batch_run["target_fetch"]["docs_by_canonical_id"]
        for canonical_id in canonical_ids:
            doc = batch_docs.get(canonical_id)
            if doc is not None and canonical_id not in docs_by_canonical_id:
                docs_by_canonical_id[canonical_id] = doc
    return docs_by_canonical_id


def collect_baseline_docs_by_canonical_id(
    result: dict[str, Any],
) -> dict[str, dict[str, Any]]:
    docs_by_canonical_id: dict[str, dict[str, Any]] = {}
    for batch_run in get_batch_runs(result):
        batch_docs = batch_run["baseline_fetch"]["docs_by_canonical_id"]
        for canonical_id, doc in batch_docs.items():
            if canonical_id not in docs_by_canonical_id:
                docs_by_canonical_id[canonical_id] = doc
    return docs_by_canonical_id


def select_source_fields(
    document: dict[str, Any],
    source_fields: list[str] | None,
) -> dict[str, Any]:
    if source_fields is None:
        return document
    return {field: document[field] for field in source_fields if field in document}


def build_scala_decode_input_records(
    docs_by_canonical_id: dict[str, dict[str, Any]],
    source_fields: list[str] | None = None,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for canonical_id, doc in docs_by_canonical_id.items():
        source_identifier = extract_root_source_identifier(doc["_source"])
        records.append(
            {
                "docId": canonical_id,
                "sourceIdentifier": (
                    build_source_identifier_string(source_identifier)
                    if source_identifier is not None
                    else None
                ),
                "document": select_source_fields(doc["_source"], source_fields),
            }
        )
    return records


def write_scala_decode_input(
    records: list[dict[str, Any]],
    mode: str,
    label: str | None = None,
    index_label: str | None = None,
) -> Path:
    suffix = uuid.uuid4().hex[:8]
    label_prefix = f"{label}-" if label else ""
    index_part = f"-{index_label}" if index_label else ""
    input_path = SCALA_DECODE_DIR / f"{label_prefix}{mode}{index_part}-{suffix}.jsonl"
    lines = [json.dumps(record) for record in records]
    input_path.write_text("\n".join(lines) + ("\n" if lines else ""))
    return input_path


def run_scala_decode_checker(
    mode: str,
    input_path: Path,
    output_path: Path | None = None,
) -> dict[str, Any]:
    output_path = output_path or SCALA_DECODE_DIR / f"{input_path.stem}-summary.json"
    command = [
        "sbt",
        "project merger",
        (
            f"runMain {SCALA_DECODE_MAIN_CLASS} "
            f"--mode {mode} --input {input_path} --output {output_path}"
        ),
    ]
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            "Scala decode checker failed.\n"
            f"Command: {' '.join(command)}\n"
            f"Exit code: {completed.returncode}\n"
            f"STDOUT:\n{completed.stdout[-4000:]}\n"
            f"STDERR:\n{completed.stderr[-4000:]}"
        )

    summary = json.loads(output_path.read_text())
    return {
        "mode": mode,
        "input_path": input_path,
        "output_path": output_path,
        "summary": summary,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }


def run_scala_decode_from_result(
    result: dict[str, Any],
    mode: str,
    label: str | None = None,
) -> dict[str, Any]:
    canonical_ids = collect_manifest_canonical_ids(result)
    docs_by_canonical_id = collect_target_docs_by_canonical_id(result)
    source_fields = MATCHER_SOURCE_FIELDS if mode == "matcher" else None
    missing_canonical_ids = [
        canonical_id for canonical_id in canonical_ids if canonical_id not in docs_by_canonical_id
    ]
    records = build_scala_decode_input_records(
        docs_by_canonical_id,
        source_fields=source_fields,
    )
    input_path = write_scala_decode_input(
        records, mode=mode, label=label,
        index_label=f"target-{TARGET_INDEX_DATE_SUFFIX}",
    )
    decode_result = run_scala_decode_checker(mode=mode, input_path=input_path)
    decode_result["record_count"] = len(records)
    decode_result["missing_canonical_ids"] = missing_canonical_ids
    decode_result["reused_target_docs"] = True
    return decode_result


def decode_failures_frame(decode_result: dict[str, Any]) -> Any:
    failures = decode_result["summary"].get("failures", [])
    if pd is None:
        return failures
    return pd.DataFrame(failures)


def build_scala_compare_records(
    target_docs_by_id: dict[str, dict[str, Any]],
    baseline_docs_by_id: dict[str, dict[str, Any]],
) -> tuple[list[dict[str, Any]], dict[str, int]]:
    paired_ids = sorted(set(target_docs_by_id) & set(baseline_docs_by_id))
    target_only = sorted(set(target_docs_by_id) - set(baseline_docs_by_id))
    baseline_only = sorted(set(baseline_docs_by_id) - set(target_docs_by_id))
    records: list[dict[str, Any]] = []
    for canonical_id in paired_ids:
        target_doc = target_docs_by_id[canonical_id]
        baseline_doc = baseline_docs_by_id[canonical_id]
        source_identifier = extract_root_source_identifier(target_doc["_source"])
        records.append(
            {
                "docId": canonical_id,
                "sourceIdentifier": (
                    build_source_identifier_string(source_identifier)
                    if source_identifier is not None
                    else None
                ),
                "targetDocument": target_doc["_source"],
                "baselineDocument": baseline_doc["_source"],
            }
        )
    stats = {
        "paired": len(paired_ids),
        "target_only": len(target_only),
        "baseline_only": len(baseline_only),
    }
    return records, stats


def run_scala_decode_comparison(
    result: dict[str, Any],
    label: str | None = None,
) -> dict[str, Any]:
    target_docs = collect_target_docs_by_canonical_id(result)
    baseline_docs = collect_baseline_docs_by_canonical_id(result)

    records, pairing_stats = build_scala_compare_records(target_docs, baseline_docs)
    input_path = write_scala_decode_input(
        records, mode="compare", label=label,
        index_label=f"target-{TARGET_INDEX_DATE_SUFFIX}_vs_baseline-{PIPELINE_DATE}",
    )
    compare_result = run_scala_decode_checker(mode="compare", input_path=input_path)
    compare_result["record_count"] = len(records)
    compare_result["pairing_stats"] = pairing_stats
    return compare_result

In [13]:
# Re-run this cell after changing the Python id_minter so `verify_1000`
# reflects freshly written identified docs rather than an older batch result.

candidate_pool = sample_source_work_candidates(
    desired_count=1 + VERIFICATION_BATCH_SIZE,
    oversample_buffer=max(DEFAULT_SAMPLE_BUFFER, VERIFICATION_BATCH_SIZE // 4),
)

# candidate_pool[:3]

# Smoke test:
# smoke = run_single_batch(
#     candidate_pool[:1],
#     label="smoke",
# )
# smoke["summary"]

# Verification batch of 1k:
verify_1000 = run_single_batch(
    candidate_pool[1 : 1 + VERIFICATION_BATCH_SIZE],
    label="verify-1000",
)
verify_1000["summary"]

2026-03-13 07:57:25 [info     ] Creating Elasticsearch client  es_mode=public host_config=https://0687c4947f404fe8ae10fa1aeab71bdc.eu-west-1.aws.found.io:443
Sampled 1001 candidates from works-source-2025-10-02 (seed=1341584205, skipped={})
2026-03-13 07:57:57 [info     ] Creating Elasticsearch client  es_mode=public host_config=https://0687c4947f404fe8ae10fa1aeab71bdc.eu-west-1.aws.found.io:443
2026-03-13 07:57:58 [info     ] Creating Elasticsearch client  es_mode=public host_config=https://0687c4947f404fe8ae10fa1aeab71bdc.eu-west-1.aws.found.io:443


{'job_id': 'id-minter-verify-20260313T075725Z-verify-1000-b80fed1a',
 'batch_size': 1000,
 'status_code': 200,
 'function_error': None,
 'lambda_seconds': 30.837,
 'manifest_seconds': 0.098,
 'target_fetch_seconds': 1.068,
 'baseline_fetch_seconds': 1.258,
 'total_post_lambda_seconds': 2.441,
 'manifest_success_count': 1000,
 'manifest_failure_count': 0,
 'success_batch_line_count': 1,
 'failure_batch_line_count': 0,
 'target_manifest_canonical_ids': 1000,
 'target_docs_found': 1000,
 'target_missing_docs': 0,
 'baseline_docs_found': 1000,
 'baseline_missing_docs': 0}

In [14]:
# Scala matcher decode check against docs written to the new identified index.
# This reuses docs already downloaded during comparison in `run_single_batch`.
# Use a freshly rerun batch result, for example `smoke` or `verify_1000`,
# after re-running the batch cell above.

decode_source = verify_1000
matcher_decode = run_scala_decode_from_result(
    decode_source,
    mode="matcher",
    label="matcher-decode",
)
matcher_decode["summary"]
decode_failures_frame(matcher_decode)


""


In [15]:
# Scala merger decode check against full docs from the new identified index.
# This is a stricter compatibility check than matcher mode because it decodes full Work[Identified].
# It reuses the full target docs already fetched during comparison.
# Use a freshly rerun batch result, for example `smoke` or `verify_1000`,
# after re-running the batch cell above.

decode_source = verify_1000
merger_decode = run_scala_decode_from_result(
    decode_source,
    mode="merger",
    label="merger-decode",
)
merger_decode["summary"]
decode_failures_frame(merger_decode)


""


In [16]:
# Scala decode comparison: download full target and baseline docs, decode both
# as Work[Identified], and check they deserialise to the same Scala object.
# This proves the extra type fields / explicit nulls in Python output are
# transparent to downstream Scala consumers.

compare_result = run_scala_decode_comparison(verify_1000, label="compare")
print(f"Pairing stats: {compare_result['pairing_stats']}")
compare_result["summary"]

Pairing stats: {'paired': 1000, 'target_only': 0, 'baseline_only': 0}


{'mode': 'compare',
 'total': 1000,
 'matched': 1000,
 'failed': 0,
 'failures': []}

In [65]:
# Invoke the lambda for a single work by canonical ID, then compare.
# Set WORK_ID below and run this cell.

WORK_ID = "t9fzaa4w"  # e.g. "abc12345"

if not WORK_ID:
    print("Set WORK_ID above and re-run this cell.")
else:
    # Look up the baseline doc to get the source identifier string
    baseline_fetch = fetch_docs_by_canonical_id(
        get_baseline_es_client(), BASELINE_IDENTIFIED_INDEX, [WORK_ID],
    )
    baseline_docs = baseline_fetch["docs_by_canonical_id"]

    if WORK_ID not in baseline_docs:
        print(f"Work {WORK_ID} not found in baseline index {BASELINE_IDENTIFIED_INDEX}")
    else:
        source_identifier = extract_root_source_identifier(baseline_docs[WORK_ID]["_source"])
        if source_identifier is None:
            print(f"Work {WORK_ID} has no sourceIdentifier in baseline doc")
        else:
            source_id_str = build_source_identifier_string(source_identifier)
            print(f"Source identifier: {source_id_str}")

            # Invoke the lambda to mint this single ID
            invocation = invoke_id_minter([source_id_str], label=f"single-{WORK_ID}")
            print(f"Lambda status: {invocation['status_code']}, error: {invocation['function_error']}")
            if invocation["function_error"]:
                print(f"Lambda error payload:\n{invocation['raw_payload']}")
            else:
                # Fetch the freshly written target doc and compare
                target_fetch = fetch_docs_by_canonical_id(
                    get_target_es_client(), TARGET_IDENTIFIED_INDEX, [WORK_ID],
                )
                target_docs = target_fetch["docs_by_canonical_id"]

                if WORK_ID not in target_docs:
                    print(f"Work {WORK_ID} not found in target index {TARGET_IDENTIFIED_INDEX} after lambda invocation")
                else:
                    records, pairing_stats = build_scala_compare_records(target_docs, baseline_docs)
                    input_path = write_scala_decode_input(
                        records, mode="compare", label="single",
                        index_label=f"target-{TARGET_INDEX_DATE_SUFFIX}_vs_baseline-{PIPELINE_DATE}",
                    )
                    result = run_scala_decode_checker(mode="compare", input_path=input_path)
                    print(f"Work {WORK_ID}:")
                    print(json.dumps(result["summary"], indent=2))

Source identifier: Work[ebsco-alt-lookup/ebs206677e]
Lambda status: 200, error: None
Work t9fzaa4w:
{
  "mode": "compare",
  "total": 1,
  "matched": 1,
  "failed": 0,
  "failures": []
}


In [67]:
## Full-index batch processing

# --- Scan ---

from concurrent.futures import ThreadPoolExecutor, Future, wait, FIRST_COMPLETED
from itertools import islice
import subprocess

import elasticsearch.helpers


def scan_all_source_candidates(
    max_candidates: int | None = None,
) -> list[dict[str, Any]]:
    """Scan all documents from the source index, returning candidate dicts.

    No date filtering — all documents are included. Deduplicates by
    source_identifier_str. If max_candidates is set, stops early once
    that many candidates are collected.
    """
    es_client = get_source_es_client()
    scanner = elasticsearch.helpers.scan(
        es_client,
        index=SOURCE_INDEX,
        query={"query": {"match_all": {}}},
        _source=["state.sourceIdentifier", "state.sourceModifiedTime"],
        scroll="5m",
        size=2000,
    )

    candidates: list[dict[str, Any]] = []
    skipped: Counter = Counter()
    seen: set[str] = set()
    log_interval = 50_000

    for hit in scanner:
        state = hit["_source"].get("state", {})
        source_identifier = state.get("sourceIdentifier")
        if not source_identifier:
            skipped["missing_root_source_identifier"] += 1
            continue

        source_identifier_str = build_source_identifier_string(source_identifier)
        if source_identifier_str in seen:
            skipped["duplicate_root_source_identifier"] += 1
            continue

        seen.add(source_identifier_str)
        candidates.append({
            "source_doc_id": hit["_id"],
            "source_identifier": source_identifier,
            "source_identifier_str": source_identifier_str,
            "source_modified_time": state.get("sourceModifiedTime", ""),
        })

        if len(candidates) % log_interval == 0:
            print(f"  scanned {len(candidates):,} candidates so far...")

        if max_candidates is not None and len(candidates) >= max_candidates:
            break

    print(
        f"Scanned {len(candidates):,} candidates from {SOURCE_INDEX} "
        f"(skipped={dict(skipped)})"
    )
    return candidates


# --- Candidate persistence ---

def save_candidates(run_dir: Path, candidates: list[dict[str, Any]]) -> Path:
    path = run_dir / "candidates.jsonl"
    with path.open("w") as f:
        for c in candidates:
            f.write(json.dumps(c) + "\n")
    print(f"Saved {len(candidates):,} candidates to {path}")
    return path


def load_candidates(run_dir: Path) -> list[dict[str, Any]]:
    path = run_dir / "candidates.jsonl"
    candidates = []
    with path.open() as f:
        for line in f:
            if line.strip():
                candidates.append(json.loads(line))
    print(f"Loaded {len(candidates):,} candidates from {path}")
    return candidates


# --- Checkpoint ---

def save_checkpoint(run_dir: Path, batch_index: int, summary: dict[str, Any]) -> None:
    path = run_dir / "progress.jsonl"
    record = {
        "batch_index": batch_index,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        **summary,
    }
    with path.open("a") as f:
        f.write(json.dumps(record) + "\n")


def load_checkpoint(run_dir: Path) -> tuple[set[int], list[dict[str, Any]]]:
    path = run_dir / "progress.jsonl"
    completed: set[int] = set()
    summaries: list[dict[str, Any]] = []
    if path.exists():
        with path.open() as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    completed.add(record["batch_index"])
                    summaries.append(record)
    return completed, summaries


# --- Diff persistence ---

def save_batch_diffs(
    run_dir: Path, batch_index: int, diff_records: list[dict[str, Any]],
) -> Path:
    diffs_dir = run_dir / "diffs"
    diffs_dir.mkdir(parents=True, exist_ok=True)
    path = diffs_dir / f"batch_{batch_index:04d}.jsonl"
    with path.open("w") as f:
        for record in diff_records:
            f.write(json.dumps(record) + "\n")
    return path


def load_batch_diffs(run_dir: Path, batch_index: int) -> list[dict[str, Any]]:
    path = run_dir / "diffs" / f"batch_{batch_index:04d}.jsonl"
    if not path.exists():
        return []
    records = []
    with path.open() as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


# --- Phase 1: process one completed lambda invocation (NO SBT) ---

def process_lambda_result(
    batch_index: int,
    invocation: dict[str, Any],
    run_dir: Path,
) -> dict[str, Any]:
    """After a lambda invocation completes, fetch docs, write compare JSONL, checkpoint.

    Does NOT run SBT — that happens in Phase 2 as a single consolidated run.
    """
    started = perf_counter()

    if invocation["function_error"]:
        summary = {
            "batch_index": batch_index,
            "status": "lambda_error",
            "function_error": invocation["function_error"],
            "lambda_seconds": round(invocation["elapsed_seconds"], 3),
            "compare_input_records": 0,
            "docs_processed": 0,
        }
        save_checkpoint(run_dir, batch_index, summary)
        return summary

    manifest_batches = read_manifest_batches(invocation["manifest"])
    target_canonical_ids = flatten_manifest_canonical_ids(manifest_batches)

    target_fetch = fetch_docs_by_canonical_id(
        get_target_es_client(), TARGET_IDENTIFIED_INDEX, target_canonical_ids,
    )
    baseline_fetch = fetch_docs_by_canonical_id(
        get_baseline_es_client(), BASELINE_IDENTIFIED_INDEX, target_canonical_ids,
    )

    target_docs = target_fetch["docs_by_canonical_id"]
    baseline_docs = baseline_fetch["docs_by_canonical_id"]

    records, pairing_stats = build_scala_compare_records(target_docs, baseline_docs)

    # Write compare input JSONL for Phase 2 (no SBT call here)
    compare_input_dir = run_dir / "compare_inputs"
    compare_input_dir.mkdir(parents=True, exist_ok=True)
    compare_input_path = compare_input_dir / f"batch_{batch_index:04d}.jsonl"
    with compare_input_path.open("w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

    manifest_success = (
        invocation["manifest"].successes.count
        if invocation["manifest"] is not None else 0
    )
    manifest_failure = (
        invocation["manifest"].failures.count
        if invocation["manifest"] is not None
        and invocation["manifest"].failures is not None
        else 0
    )

    summary = {
        "batch_index": batch_index,
        "status": "ok",
        "lambda_seconds": round(invocation["elapsed_seconds"], 3),
        "post_lambda_seconds": round(perf_counter() - started, 3),
        "docs_processed": manifest_success,
        "manifest_failures": manifest_failure,
        "target_docs_found": len(target_docs),
        "baseline_docs_found": len(baseline_docs),
        "baseline_missing": len(baseline_fetch["missing_canonical_ids"]),
        "paired": pairing_stats["paired"],
        "target_only": pairing_stats["target_only"],
        "baseline_only": pairing_stats["baseline_only"],
        "compare_input_records": len(records),
    }

    save_checkpoint(run_dir, batch_index, summary)
    return summary


# --- Phase 2 helpers ---

def build_compare_chunks(
    run_dir: Path,
    chunk_size: int = 50_000,
) -> tuple[list[Path], dict[str, int]]:
    """Split per-batch compare JSONL files into chunks for separate SBT runs.

    Returns (chunk_paths, doc_to_batch) where doc_to_batch maps docId -> batch_index.
    """
    compare_input_dir = run_dir / "compare_inputs"
    chunks_dir = run_dir / "compare_chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)
    doc_to_batch: dict[str, int] = {}
    chunk_paths: list[Path] = []
    total_lines = 0
    chunk_idx = 0
    current_chunk_lines = 0
    current_chunk_file = None

    for batch_file in sorted(compare_input_dir.glob("batch_*.jsonl")):
        batch_idx = int(batch_file.stem.split("_")[1])
        with batch_file.open() as inp:
            for line in inp:
                if not line.strip():
                    continue
                if current_chunk_file is None or current_chunk_lines >= chunk_size:
                    if current_chunk_file is not None:
                        current_chunk_file.close()
                    chunk_path = chunks_dir / f"chunk_{chunk_idx:04d}.jsonl"
                    chunk_paths.append(chunk_path)
                    current_chunk_file = chunk_path.open("w")
                    chunk_idx += 1
                    current_chunk_lines = 0
                record = json.loads(line)
                doc_to_batch[record.get("docId", "")] = batch_idx
                current_chunk_file.write(line)
                current_chunk_lines += 1
                total_lines += 1

    if current_chunk_file is not None:
        current_chunk_file.close()

    print(
        f"Split {total_lines:,} records into {len(chunk_paths)} chunks "
        f"(max {chunk_size:,} per chunk) in {chunks_dir}"
    )
    return chunk_paths, doc_to_batch


def run_scala_compare_with_progress(
    input_path: Path,
    output_path: Path,
) -> dict[str, Any]:
    """Run the Scala IdentifiedIndexDecodeChecker in compare mode via SBT.

    Streams stdout live so you can monitor progress.
    Returns the parsed compare summary JSON.
    """
    sbt_run_cmd = (
        f"runMain {SCALA_DECODE_MAIN_CLASS} "
        f"--mode compare --input {input_path} --output {output_path}"
    )
    full_cmd = ["sbt", "-J-Xmx8g", "project merger", sbt_run_cmd]

    print(f"Running SBT compare: {' '.join(full_cmd)}")
    print(f"Input:  {input_path}")
    print(f"Output: {output_path}")
    print("--- SBT output ---")

    proc = subprocess.Popen(
        full_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        cwd=str(REPO_ROOT),
        text=True,
        bufsize=1,
    )

    info_lines = 0
    warn_lines = 0
    error_lines = 0

    for line in proc.stdout:
        line = line.rstrip()
        if line.startswith("[info]"):
            info_lines += 1
            # Print periodic progress
            if info_lines % 100 == 0 or "compared" in line.lower() or "done" in line.lower():
                print(f"  {line}")
        elif line.startswith("[error]"):
            error_lines += 1
            print(f"  {line}")
        elif line.startswith("[warn]"):
            warn_lines += 1
        else:
            # Print non-SBT output (e.g. download progress)
            if line.strip():
                print(f"  {line}")

    proc.wait()
    print(f"--- SBT finished (exit={proc.returncode}, info={info_lines}, warn={warn_lines}, error={error_lines}) ---")

    if output_path.exists():
        return json.loads(output_path.read_text())
    else:
        print(f"WARNING: output file not found at {output_path}")
        return {"total": 0, "matched": 0, "failed": 0, "failures": []}


def distribute_compare_failures(
    compare_summary: dict[str, Any],
    doc_to_batch: dict[str, int],
    run_dir: Path,
) -> dict[int, int]:
    """Distribute compare failures from the single SBT run back to per-batch diff files.

    Returns dict of batch_index -> number of failures.
    """
    failures = compare_summary.get("failures", [])
    if not failures:
        print("No failures to distribute.")
        return {}

    batch_failures: dict[int, list[dict[str, Any]]] = {}
    unmatched = 0
    for failure in failures:
        doc_id = failure.get("docId", "")
        batch_idx = doc_to_batch.get(doc_id)
        if batch_idx is not None:
            batch_failures.setdefault(batch_idx, []).append(failure)
        else:
            unmatched += 1

    diff_counts: dict[int, int] = {}
    for batch_idx, batch_diffs in sorted(batch_failures.items()):
        save_batch_diffs(run_dir, batch_idx, batch_diffs)
        diff_counts[batch_idx] = len(batch_diffs)
        print(f"  Batch {batch_idx:4d}: {len(batch_diffs)} failure(s)")

    if unmatched:
        print(f"  ({unmatched} failures could not be mapped to a batch)")

    print(f"Distributed {len(failures)} failures across {len(batch_failures)} batches")
    return diff_counts


print("Full-run helper functions defined (two-phase).")

Full-run helper functions defined (two-phase).


In [ ]:
# Full-index orchestrator (Phase 1).
# Lambda invocations with sliding-window parallelism.
# Each completed lambda triggers doc fetch + compare JSONL write (no SBT).
# Re-running continues from the last completed batch.

FULL_RUN_BATCH_SIZE = 1000
MAX_CONCURRENCY = 20
MAX_CANDIDATES = None    # set to None for full index; e.g. 5000 for testing
MAX_BATCHES = None       # set to e.g. 2 to limit batches processed per execution
FULL_RUN_ID = RUN_PREFIX # paste a previous run ID string to resume that run

# --- Resolve run directory ---
FULL_RUN_DIR = SCALA_DECODE_DIR / "full_run" / FULL_RUN_ID
FULL_RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run ID:       {FULL_RUN_ID}")
print(f"Run dir:      {FULL_RUN_DIR}")
print(f"Batch size:   {FULL_RUN_BATCH_SIZE}")
print(f"Concurrency:  {MAX_CONCURRENCY}")
print(f"Max cands:    {MAX_CANDIDATES or 'all'}")
print(f"Max batches:  {MAX_BATCHES or 'all'}")
print()

# --- Load or scan candidates ---
candidates_path = FULL_RUN_DIR / "candidates.jsonl"
if candidates_path.exists():
    all_candidates = load_candidates(FULL_RUN_DIR)
else:
    print("Scanning source index for candidates...")
    all_candidates = scan_all_source_candidates(max_candidates=MAX_CANDIDATES)
    save_candidates(FULL_RUN_DIR, all_candidates)

# --- Chunk into batches ---
total_batches = len(all_candidates) // FULL_RUN_BATCH_SIZE
if MAX_BATCHES is not None:
    total_batches = min(total_batches, MAX_BATCHES)

print(f"\nTotal candidates: {len(all_candidates):,}")
print(f"Total batches:    {total_batches}")

# --- Load checkpoint ---
completed_indices, completed_summaries = load_checkpoint(FULL_RUN_DIR)
print(f"Already completed: {len(completed_indices)} batches")

# --- Determine remaining work ---
remaining_indices = [i for i in range(total_batches) if i not in completed_indices]
print(f"Remaining:         {len(remaining_indices)} batches")

if not remaining_indices:
    print("\nAll batches complete!")
else:
    run_started = perf_counter()

    def _get_batch_source_ids(batch_idx: int) -> list[str]:
        start = batch_idx * FULL_RUN_BATCH_SIZE
        batch_candidates = all_candidates[start : start + FULL_RUN_BATCH_SIZE]
        return [c["source_identifier_str"] for c in batch_candidates]

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENCY) as executor:
        # Sliding window: submit up to MAX_CONCURRENCY futures, then
        # as each completes, immediately submit the next one.
        pending: set[Future] = set()
        future_to_index: dict[Future, int] = {}
        remaining_iter = iter(remaining_indices)

        # Fill initial window
        for batch_idx in islice(remaining_iter, MAX_CONCURRENCY):
            fut = executor.submit(invoke_id_minter, _get_batch_source_ids(batch_idx), label=f"full-{batch_idx:04d}")
            pending.add(fut)
            future_to_index[fut] = batch_idx

        while pending:
            # Wait for the first completion
            done, pending = wait(pending, return_when=FIRST_COMPLETED)

            for fut in done:
                batch_idx = future_to_index.pop(fut)
                invocation = fut.result()

                # Process immediately (fetch docs, write compare JSONL, checkpoint)
                summary = process_lambda_result(batch_idx, invocation, FULL_RUN_DIR)

                done_so_far = len(completed_indices) + 1
                completed_indices.add(batch_idx)
                elapsed = perf_counter() - run_started

                status_icon = "ok" if summary["status"] == "ok" else "ERR"
                records_info = f", {summary['compare_input_records']} compare records" if summary.get("compare_input_records") else ""
                baseline_missing = (
                    f", {summary['baseline_missing']} baseline-missing"
                    if summary.get("baseline_missing") else ""
                )

                print(
                    f"  Batch {batch_idx:4d}/{total_batches}: "
                    f"[{status_icon}] "
                    f"{summary.get('docs_processed', 0)} docs"
                    f"{records_info}{baseline_missing} "
                    f"({summary.get('lambda_seconds', 0):.1f}s lambda, "
                    f"{summary.get('post_lambda_seconds', 0):.1f}s post) "
                    f"[{done_so_far}/{total_batches} done, {elapsed:.0f}s elapsed]"
                )

            # Backfill: for each completed future, submit the next batch
            for _ in range(len(done)):
                next_idx = next(remaining_iter, None)
                if next_idx is not None:
                    fut = executor.submit(invoke_id_minter, _get_batch_source_ids(next_idx), label=f"full-{next_idx:04d}")
                    pending.add(fut)
                    future_to_index[fut] = next_idx

    total_elapsed = perf_counter() - run_started
    print(f"\nPhase 1 finished: {len(remaining_indices)} batches in {total_elapsed:.1f}s")
    print("Run Phase 2 cell next for Scala comparison.")


Run ID:       id-minter-verify-20260313T103459Z
Run dir:      /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T103459Z
Batch size:   1000
Concurrency:  20
Max cands:    all
Max batches:  all

Scanning source index for candidates...
2026-03-13 10:37:20 [info     ] Creating Elasticsearch client  es_mode=public host_config=https://0687c4947f404fe8ae10fa1aeab71bdc.eu-west-1.aws.found.io:443
  scanned 50,000 candidates so far...
  scanned 100,000 candidates so far...
  scanned 150,000 candidates so far...
  scanned 200,000 candidates so far...
  scanned 250,000 candidates so far...
  scanned 300,000 candidates so far...
  scanned 350,000 candidates so far...
  scanned 400,000 candidates so far...
  scanned 450,000 candidates so far...
  scanned 500,000 candidates so far...
  scanned 550,000 candidates so far...
  scanned 600,000 candidates so far...
  scanned 650,000 candidates so far...
  scanned 700,000 candidates so far...
  s

In [68]:
# Phase 2: Scala decode comparison.
# Splits compare inputs into chunks (50k records each), runs SBT on each chunk,
# then merges results and distributes failures to per-batch diff files.
# This avoids JVM OOM when processing large datasets.

COMPARE_CHUNK_SIZE = 50_000  # records per SBT invocation (50k proven to work with 8GB heap)

print(f"Run dir:    {FULL_RUN_DIR}")
print(f"Chunk size: {COMPARE_CHUNK_SIZE:,}\n")

compare_input_dir = FULL_RUN_DIR / "compare_inputs"
if not compare_input_dir.exists() or not any(compare_input_dir.glob("batch_*.jsonl")):
    print("No compare inputs found. Run Phase 1 first.")
else:
    # --- Split into chunks ---
    chunk_paths, doc_to_batch = build_compare_chunks(FULL_RUN_DIR, chunk_size=COMPARE_CHUNK_SIZE)

    # --- Run SBT on each chunk ---
    all_failures: list[dict[str, Any]] = []
    total_compared = 0
    total_matched = 0
    total_failed = 0

    for i, chunk_path in enumerate(chunk_paths):
        chunk_output = chunk_path.with_suffix(".summary.json")
        # Skip chunks that already have a summary (resumable)
        if chunk_output.exists():
            chunk_summary = json.loads(chunk_output.read_text())
            print(f"Chunk {i + 1}/{len(chunk_paths)}: already done — "
                  f"{chunk_summary['matched']}/{chunk_summary['total']} matched")
        else:
            print(f"\n{'='*60}")
            print(f"Chunk {i + 1}/{len(chunk_paths)}: {chunk_path.name}")
            print(f"{'='*60}")
            chunk_summary = run_scala_compare_with_progress(chunk_path, chunk_output)

        total_compared += chunk_summary.get("total", 0)
        total_matched += chunk_summary.get("matched", 0)
        total_failed += chunk_summary.get("failed", 0)
        all_failures.extend(chunk_summary.get("failures", []))

    # --- Write merged summary ---
    compare_summary = {
        "mode": "compare",
        "total": total_compared,
        "matched": total_matched,
        "failed": total_failed,
        "failures": all_failures,
    }
    output_path = FULL_RUN_DIR / "compare_summary.json"
    output_path.write_text(json.dumps(compare_summary, indent=2))

    # --- Distribute failures ---
    print()
    diff_counts = distribute_compare_failures(compare_summary, doc_to_batch, FULL_RUN_DIR)

    # --- Final summary ---
    print(f"\nCompare summary saved to {output_path}")
    print(
        f"Total: {total_compared:,} compared, "
        f"{total_matched:,} matched, "
        f"{total_failed:,} failed "
        f"(across {len(chunk_paths)} chunks)"
    )

Run dir:    /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T095546Z
Chunk size: 50,000

Split 500,000 records into 10 chunks (max 50,000 per chunk) in /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T095546Z/compare_chunks

Chunk 1/10: chunk_0000.jsonl
Running SBT compare: sbt -J-Xmx8g project merger runMain weco.pipeline.merger.tools.IdentifiedIndexDecodeChecker --mode compare --input /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T095546Z/compare_chunks/chunk_0000.jsonl --output /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T095546Z/compare_chunks/chunk_0000.summary.json
Input:  /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T095546Z/compare_chunks/chunk_0000.jsonl
Output: /

In [69]:
# Summary: read Phase 1 progress and Phase 2 compare results.
# Run this after both phases complete (or after Phase 1 for lambda stats only).

# Use the run dir from the orchestrator cell, or set manually:
# FULL_RUN_DIR = SCALA_DECODE_DIR / "full_run" / "<your-run-id>"

completed_indices, summaries = load_checkpoint(FULL_RUN_DIR)

total_docs = sum(s.get("docs_processed", 0) for s in summaries)
total_compare_records = sum(s.get("compare_input_records", 0) for s in summaries)
total_lambda_errors = sum(1 for s in summaries if s.get("status") != "ok")
total_baseline_missing = sum(s.get("baseline_missing", 0) for s in summaries)
avg_lambda_secs = (
    sum(s.get("lambda_seconds", 0) for s in summaries) / len(summaries)
    if summaries else 0
)

print(f"Run directory:          {FULL_RUN_DIR}")
print(f"Batches completed:      {len(summaries)}")
print(f"Total docs processed:   {total_docs:,}")
print(f"Compare records:        {total_compare_records:,}")
print(f"Lambda errors:          {total_lambda_errors}")
print(f"Baseline-missing total: {total_baseline_missing}")
print(f"Avg lambda time:        {avg_lambda_secs:.2f}s")

# Phase 2 compare results
compare_path = FULL_RUN_DIR / "compare_summary.json"
if compare_path.exists():
    compare_summary = json.loads(compare_path.read_text())
    print(
        f"\nScala comparison:       {compare_summary['total']} total, "
        f"{compare_summary['matched']} matched, "
        f"{compare_summary['failed']} failed"
    )
else:
    print("\nScala comparison:       not yet run (run Phase 2 cell)")

# Per-batch diffs
diffs_dir = FULL_RUN_DIR / "diffs"
if diffs_dir.exists():
    diff_files = sorted(diffs_dir.glob("batch_*.jsonl"))
    if diff_files:
        print(f"\nBatches with diffs ({len(diff_files)}):")
        for diff_file in diff_files:
            batch_idx = int(diff_file.stem.split("_")[1])
            count = sum(1 for line in diff_file.open() if line.strip())
            print(f"  batch {batch_idx:4d}: {count} diffs")

if pd is not None:
    df = pd.DataFrame(summaries)
    display(df)

Run directory:          /Users/kennyr/workspace/catalogue-pipeline/.scratch/id_minter_decode_checks/full_run/id-minter-verify-20260313T095546Z
Batches completed:      500
Total docs processed:   500,000
Compare records:        500,000
Lambda errors:          0
Baseline-missing total: 0
Avg lambda time:        23.48s

Scala comparison:       500000 total, 500000 matched, 0 failed


,batch_index,timestamp,status,lambda_seconds,post_lambda_seconds,docs_processed,manifest_failures,target_docs_found,baseline_docs_found,baseline_missing,paired,target_only,baseline_only,compare_input_records
0,14,2026-03-13T09:57:21.314426+00:00,ok,25.715,0.860,1000,0,1000,1000,0,1000,0,0,1000
1,5,2026-03-13T09:57:22.096567+00:00,ok,25.769,0.776,1000,0,1000,1000,0,1000,0,0,1000
2,9,2026-03-13T09:57:22.808077+00:00,ok,27.216,0.705,1000,0,1000,1000,0,1000,0,0,1000
3,6,2026-03-13T09:57:23.463445+00:00,ok,26.800,0.648,1000,0,1000,1000,0,1000,0,0,1000
4,1,2026-03-13T09:57:24.547060+00:00,ok,29.133,0.680,1000,0,1000,1000,0,1000,0,0,1000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,2026-03-13T10:11:44.989529+00:00,ok,23.612,0.742,1000,0,1000,1000,0,1000,0,0,1000
496,496,2026-03-13T10:11:47.422060+00:00,ok,23.971,0.733,1000,0,1000,1000,0,1000,0,0,1000
497,497,2026-03-13T10:11:48.108229+00:00,ok,23.666,0.680,1000,0,1000,1000,0,1000,0,0,1000
498,498,2026-03-13T10:11:48.814441+00:00,ok,22.632,0.698,1000,0,1000,1000,0,1000,0,0,1000


In [71]:
# Diff review: iterate through all batches that have diffs.

_, all_summaries = load_checkpoint(FULL_RUN_DIR)
batches_with_diffs = [s for s in all_summaries if s.get("diff_count", 0) > 0]

if not batches_with_diffs:
    print("No batches have diffs.")
else:
    all_diff_records: list[dict[str, Any]] = []
    print(f"{len(batches_with_diffs)} batch(es) with diffs:\n")

    for batch_summary in batches_with_diffs:
        batch_idx = batch_summary["batch_index"]
        diff_records = load_batch_diffs(FULL_RUN_DIR, batch_idx)
        print(f"── Batch {batch_idx} ({len(diff_records)} diff(s)) ──")

        for i, record in enumerate(diff_records):
            doc_id = record.get("docId", "?")
            source_id = record.get("sourceIdentifier", "?")
            error = record.get("error", record.get("message", "unknown"))
            print(f"  [{i}] {doc_id}  ({source_id})")
            print(f"      {error}")

        print()
        for r in diff_records:
            r["batch_index"] = batch_idx
        all_diff_records.extend(diff_records)

    print(f"Total: {len(all_diff_records)} diff(s) across {len(batches_with_diffs)} batch(es)")

    if pd is not None:
        display(pd.DataFrame(all_diff_records))

No batches have diffs.
